# Updates the MIRI WFSS Photom reference file to have subarray column and wfss_photmjsr.unit &B wave_out_pm.unit

In [10]:
import astropy.io.fits as fits
import astropy.units as u
from glob import glob
import numpy as np
import pdb
from pylab import *
import matplotlib as mpl
mpl.use('tkagg')
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from scipy import signal, interpolate
from stdatamodels.jwst import datamodels
import copy
from datetime import datetime
import pickle
from astropy.table import Table


In [11]:
## ==== Program and observation information ====
data_dir   = "/Users/morrison/MIRI_WFSS/Ref_Files/PHOTOM/"
spec_dir   = "/Users/morrison/MIRI_WFSS/Ref_Files/PHOTOM/"

In [12]:
## ========== Target information ===============
prog_ids = ['01536', '04496', '04496']
obs_numbs = ['027', '015', '017']
target_names = ['bd60_1753', 'hd55677', 'j1757132']

In [13]:
# Get comparison calibration from latest PHOTOM ref file

file = data_dir+'jwst_miri_photom_0216.fits'
phot_file= datamodels.MirLrsPhotomModel(file)
phot_file_wfss = datamodels.MirWfssPhotomModel()
hdulist_ref = fits.open(file)
hdulist_ref.info()
data_ref = hdulist_ref['PHOTOM', 1].data
wave_comp = data_ref['wavelength'][0]*u.micron
relres_comp = data_ref['relresponse'][0]*u.dimensionless_unscaled
pixar_sr = 2.86063256542560e-13*u.steradian
pixar_a2 = 0.01217199*u.arcsec*u.arcsec
photmjsr = data_ref['photmjsr'][0]*u.MJy*(1/u.steradian)*(1/u.DN)*(u.second)
print(np.shape(photmjsr))
print(photmjsr)
photmjsr_err = data_ref['uncertainty'][0]*u.MJy*(1/u.steradian)*(1/u.DN)*(u.second)
hdulist_ref.close()

Filename: /Users/morrison/MIRI_WFSS/Ref_Files/PHOTOM/jwst_miri_photom_0216.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      52   ()      
  1  PHOTOM        1 BinTableHDU     29   1R x 8C   [12A, 15A, E, E, I, 387E, 387E, 387E]   
()
13.96003532409668 MJy s / (DN sr)


In [14]:
for i, pid in enumerate(prog_ids[:3]):
    #read x1d file
    x1d_file = f'{data_dir}jw{pid}-o{obs_numbs[i]}_t001_miri_p750l_x1d.fits'
    print(x1d_file)
    hdulist = fits.open(x1d_file)
    spec_data = hdulist['EXTRACT1D', 1].data
    wave = spec_data['WAVELENGTH']*u.micron
    #lam in microns
    flux = spec_data['FLUX']*(u.DN/u.s)
	# flux in DN/sec
    fluxerr = spec_data['FLUX_ERROR']*(u.DN/u.s)
    dw = np.abs(np.gradient(wave))
    flux_pm = flux*dw
    fluxerr_pm = fluxerr*dw

    #print("flux_pm", flux_pm)

    # Read in CALSPEC spectrum
    starfile = glob(f'{spec_dir}{target_names[i]}*slit.sp.tbl')[0]
    stardata = np.genfromtxt(starfile, skip_header=6)
    wavestar = stardata[:,0]*u.micron
	# in microns; this array is the same for all CALSPEC spectra and the default for the output in Jy
    fluxstar = stardata[:,1] * u.Jy
    fluxstar = fluxstar / (pixar_sr)   # Jy/sr
    fluxstar = fluxstar.to(u.MJy / u.sr)
    print("flux_star units are",fluxstar.unit)
    # in MJy/sr
    dwcal = np.abs(np.gradient(wavestar))
    fluxstar_pm = fluxstar*dwcal
    print("flux_star per micron units are",fluxstar_pm.unit)


    # Clean JWST spectrum
    wave_trim = wave[~np.isnan(flux)]
    # Filter out NaNs
    # Remember smmoothed flux is the observation not Calspec
    smoothed_flux = signal.savgol_filter(flux[~np.isnan(flux)], 9, 3)*(u.DN/u.s)
    smoothed_flux_pm = smoothed_flux*np.abs(np.gradient(wave_trim))
    print("flux_star per micron units are",fluxstar_pm.unit)

    # Create smoothed spectrum

    #repeat for smoothed_flux_pm
    fint = interpolate.interp1d(
        wave_trim.value,
        smoothed_flux_pm.value,
        kind='cubic',
        fill_value='extrapolate')
    smoothed_flux_pm = fint(wave.value)*smoothed_flux_pm.unit



    print("units of flux are",flux.unit)
    print("units of smoothed_flux are", smoothed_flux.unit)
    print("units of smoothed_flux_pm are", smoothed_flux_pm.unit)


    diff_pm = flux_pm -smoothed_flux_pm
    # Calculate difference

    wave_pm = 1.0*wave

    mask_pm = np.abs(diff_pm) < 0.5 * smoothed_flux_pm
    wave_pm, flux_pm, fluxerr_pm = wave[mask_pm], flux[mask_pm], fluxerr[mask_pm]
    #mask_pm = np.where(np.absolute(diff_pm) < 0.5 * smoothed_flux_pm)
    #mask_pm = np.abs(diff_pm) < 0.5 * smoothed_flux_pm
    # Filter outliers



    #wave_pm     = wave_pm[mask_pm]
    #flux_pm     = flux_pm[mask_pm]
    #fluxerr_pm  = fluxerr_pm[mask_pm]


    #Resample JWST spectrum to stellar grid but using the pm version
    fint = interpolate.interp1d(wave_pm.value[~np.isnan(flux_pm.value)], flux_pm.value[~np.isnan(flux_pm.value)], kind='cubic', fill_value='extrapolate')

    flux_resampled_pm = fint(wavestar)*flux_pm.unit

    fint = interpolate.interp1d(wave_pm.value[~np.isnan(flux_pm.value)], fluxerr_pm.value[~np.isnan(flux_pm.value)], kind='cubic',fill_value='extrapolate')
    flux_resampled_err_pm = fint(wavestar)*fluxerr_pm.unit

    # Calculate ratio

    ratio_pm = fluxstar_pm / flux_resampled_pm
    ratio_err_pm = ratio_pm * flux_resampled_err_pm / flux_resampled_pm


    print("Units of ratio_pm are", ratio_pm.unit )
    print("Units of ratio_err_pm are", ratio_err_pm.unit)

    if i == 1:

        bad_pm =np.where((wave_pm.value > 7.2) & (wave_pm.value < 7.6)) # change here 
        ratio_pm[bad_pm] = np.nan


    # Compile relative response arrays
    if i == 0:

        relres_arrays_pm = ratio_pm
        relres_err_arrays_pm = ratio_err_pm
    else:

        relres_arrays_pm = np.vstack([relres_arrays_pm, ratio_pm])
        relres_err_arrays_pm = np.vstack([relres_err_arrays_pm, ratio_pm])






/Users/morrison/MIRI_WFSS/Ref_Files/PHOTOM/jw01536-o027_t001_miri_p750l_x1d.fits
flux_star units are MJy / sr
flux_star per micron units are MJy micron / sr
flux_star per micron units are MJy micron / sr
units of flux are DN / s
units of smoothed_flux are DN / s
units of smoothed_flux_pm are DN micron / s
Units of ratio_pm are MJy micron s / (DN sr)
Units of ratio_err_pm are MJy micron s / (DN sr)
/Users/morrison/MIRI_WFSS/Ref_Files/PHOTOM/jw04496-o015_t001_miri_p750l_x1d.fits
flux_star units are MJy / sr
flux_star per micron units are MJy micron / sr
flux_star per micron units are MJy micron / sr
units of flux are DN / s
units of smoothed_flux are DN / s
units of smoothed_flux_pm are DN micron / s
Units of ratio_pm are MJy micron s / (DN sr)
Units of ratio_err_pm are MJy micron s / (DN sr)
/Users/morrison/MIRI_WFSS/Ref_Files/PHOTOM/jw04496-o017_t001_miri_p750l_x1d.fits
flux_star units are MJy / sr
flux_star per micron units are MJy micron / sr
flux_star per micron units are MJy micron

In [15]:
# Average the relres arrays across all stars

relres_pm = np.nanmean(relres_arrays_pm.value, axis=0)
relres_err_pm = np.sqrt(np.nansum(relres_err_arrays_pm**2, axis=0))/ np.sum(~np.isnan(relres_arrays_pm), axis=0)

In [16]:
# Normalize to 7 microns

fint = interpolate.interp1d(wavestar, relres_pm, kind='linear')
new_photmjsr_pm = fint(7.0)*fluxstar_pm.unit*wavestar.unit/flux.unit
fint = interpolate.interp1d(wavestar, relres_err_pm, kind='linear')
new_photmjsr_err_pm = fint(7.0)*fluxstar_pm.unit*wavestar.unit/flux.unit

In [17]:
# Invert to reflect native LRS wavelength solution


wave_out_pm = wavestar[::-1]
relres_out_pm = relres_pm[::-1] / new_photmjsr_pm
relres_err_out_pm = relres_err_pm[::-1] / new_photmjsr_pm


In [25]:
# ADNREEA below are changes I made 


#Create new ref file
nelem = len(wave_out_pm)
wfss_photmjsr = new_photmjsr_pm.value*fluxstar_pm.unit/flux.unit
wfss_photmjsr_err = new_photmjsr_err_pm.value*fluxstar_pm.unit/flux.unit


#1. Clean up your data (strip units and ensure 1D float32 arrays)
w = np.asarray(wave_out_pm.value if hasattr(wave_out_pm, 'value') else wave_out_pm, dtype='f4').flatten()
resp = np.asarray(relres_out_pm.value if hasattr(relres_out_pm, 'value') else relres_out_pm, dtype='f4').flatten()
err = np.asarray(relres_err_out_pm.value if hasattr(relres_err_out_pm, 'value') else relres_err_out_pm, dtype='f4').flatten()

# Strip units and ensure wave_out_pm etc. are plain numpy arrays
nelem = len(wave_out_pm)

print(wave_out_pm.shape)
print(relres_out_pm.shape)
print(relres_err_out_pm.shape)


new_dtype = np.dtype([
    ('filter', 'S12'),
    ('subarray', 'S15'),
    ('photmjsr', 'f4'),
    ('uncertainty', 'f4'),
    ('nelem', 'i2'), # int16 per schema
    ('wavelength', 'O'), 
    ('relresponse', 'O'),
    ('reluncertainty', 'O')
])

data_array = np.array([(
    'P750L',
    'FULL',
    float(wfss_photmjsr.value), 
    float(wfss_photmjsr_err.value), 
    len(w), 
    w, 
    resp, 
    err
)], dtype=new_dtype)


# Assign directly to the model
# The model's internal logic will handle the conversion to the FITS table format
phot_file_wfss.phot_table = data_array
out_file = 'jwst_miri_photom_WFSS_20260311.fits'

(379,)
(379,)
(379,)


In [28]:

import numpy as np
from datetime import datetime
from astropy.table import Table

def make_ref_file(phot_file, data_list, out_file):
    """
    phot_file: The JWST DataModel instance (e.g., PhotomModel)
    data_list: The list of tuples containing your calibration data
    out_file:  The string path for the output FITS file
    """

   
    # These MUST match the schema keys in your error message exactly
    names = (
        'filter',
        'subarray',
        'photmjsr', 
        'uncertainty', 
        'nelem', 
        'wavelength', 
        'relresponse', 
        'reluncertainty' # Schema says 'reluncertainty', not 'relres_err'
    )
    
    try:
        # Create the table with the explicit names from the schema
        temp_table = Table(rows=data_list, names=names)

        p_unit_str = str(wfss_photmjsr.unit)
        w_unit_str = str(wave_out_pm.unit)

        phot_file.phot_unit = p_unit_str
        phot_file.wave_unit = w_unit_str
        
        # Now the model will recognize the columns
        phot_file.phot_table = np.array(temp_table)
        print("Successfully attached table with correct schema names.")
        
    except Exception as e:
        print(f"Error attaching table: {e}")
        return


    # 2. Update Metadata
    phot_file.meta.filename = out_file
    phot_file.meta.author = 'A. Petric'
    phot_file.meta.origin = 'STScI'
    phot_file.meta.instrument.name = 'MIRI'  # Ensure instrument is set
    phot_file.meta.exposure.type = 'MIR_WFSS' 
    phot_file.meta.reftype = 'photom'
    phot_file.meta.description = "MIRI WFSS file generated during flight."
    phot_file.meta.pedigree = 'INFLIGHT 2022-07-08 2024-05-09'
    phot_file.meta.useafter = '2022-04-01T00:00:00'
    
    # Use datetime.now(timezone.utc) as utcnow() is deprecated in newer Python
    phot_file.meta.date = datetime.now().isoformat()

    # 3. Handle History
    # Instead of deleting by index which can be tricky, you can clear or just append
    phot_file.history = [] 
    phot_file.history.append("This reference file was generated from A-star slit observations modifying code from I. Wong.")
    phot_file.history.append("This was done to have per micron units as other WFSS teams")

    
    # The .save() method automatically handles the FITS HDU creation
    phot_file.save(out_file, overwrite=True)
    print(f"Successfully saved to {out_file}")

In [29]:
make_ref_file(phot_file_wfss, data_array, out_file)

Successfully attached table with correct schema names.
Successfully saved to jwst_miri_photom_WFSS_20260311.fits


/Users/morrison/miniconda3/envs/dev13/lib/python3.13/site-packages/stdatamodels/model_base.py:1133: UserWarning: The history attribute will soon be deprecated. Use add_history_entry to add history entries
  entries = self.history
/var/folders/l0/s51kjpx95j12hwymx_lfmg6h000105/T/ipykernel_77961/2849608895.py:61: UserWarning: The history attribute will soon be deprecated. Use add_history_entry to add history entries
  phot_file.history.append("This reference file was generated from A-star slit observations modifying code from I. Wong.")
/var/folders/l0/s51kjpx95j12hwymx_lfmg6h000105/T/ipykernel_77961/2849608895.py:62: UserWarning: The history attribute will soon be deprecated. Use add_history_entry to add history entries
  phot_file.history.append("This was done to have per micron units as other WFSS teams")


In [23]:
 phot_file.validate()

In [24]:
print(phot_file.phot_table.filter)


wavelength = phot_file.phot_table.wavelength


['P750L']
